In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests
from datetime import datetime, timezone
from io import StringIO
import os
import json

## Task 1.1

In [2]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

headers = {
    'User-Agent': 'Mozilla/5.0',
    'Accept': 'text/html'
}

# Fetch raw html page using request
response = requests.get(url=url,headers=headers,timeout=30)

if response.status_code == 200:

  #Convert html page to pandas table
  tables = pd.read_html(StringIO(response.text))
  sp500 = tables[0]
  display(sp500.head())

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [3]:
sp500.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Symbol                 503 non-null    object
 1   Security               503 non-null    object
 2   GICS Sector            503 non-null    object
 3   GICS Sub-Industry      503 non-null    object
 4   Headquarters Location  503 non-null    object
 5   Date added             503 non-null    object
 6   CIK                    503 non-null    int64 
 7   Founded                503 non-null    object
dtypes: int64(1), object(7)
memory usage: 31.6+ KB


In [4]:
sp500 = sp500[[
    "Symbol",
    "Security",
    "GICS Sector",
    "GICS Sub-Industry"
]].copy()
sp500

,Symbol,Security,GICS Sector,GICS Sub-Industry
0,MMM,3M,Industrials,Industrial Conglomerates
1,AOS,A. O. Smith,Industrials,Building Products
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment
3,ABBV,AbbVie,Health Care,Biotechnology
4,ACN,Accenture,Information Technology,IT Consulting & Other Services
...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment


In [5]:
# Convert Wikipedia ticker format to Yahoo Finance format
sp500["Yahoo_Ticker"] = (sp500["Symbol"].str.replace(".", "-", regex=False))

sp500["Retrieval_Date"] = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
sp500.head(10)

,Symbol,Security,GICS Sector,GICS Sub-Industry,Yahoo_Ticker,Retrieval_Date
0,MMM,3M,Industrials,Industrial Conglomerates,MMM,2026-11-24 09:10:01 UTC
1,AOS,A. O. Smith,Industrials,Building Products,AOS,2026-11-24 09:10:01 UTC
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,ABT,2026-11-24 09:10:01 UTC
3,ABBV,AbbVie,Health Care,Biotechnology,ABBV,2026-11-24 09:10:01 UTC
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,ACN,2026-11-24 09:10:01 UTC
5,ADBE,Adobe Inc.,Information Technology,Application Software,ADBE,2026-11-24 09:10:01 UTC
6,AMD,Advanced Micro Devices,Information Technology,Semiconductors,AMD,2026-11-24 09:10:01 UTC
7,AES,AES Corporation,Utilities,Independent Power Producers & Energy Traders,AES,2026-11-24 09:10:01 UTC
8,AFL,Aflac,Financials,Life & Health Insurance,AFL,2026-11-24 09:10:01 UTC
9,A,Agilent Technologies,Health Care,Life Sciences Tools & Services,A,2026-11-24 09:10:01 UTC


In [6]:
# Save the csv
os.makedirs("sp500_data", exist_ok=True)
sp500.to_csv("sp500_data/sp500_tickers.csv", index=False)

In [7]:
print("Total S&P 500 constituents:", len(sp500))

Total S&P 500 constituents: 503


In [8]:
tickers = sp500["Yahoo_Ticker"].tolist()

stock_data = {}
download_failed = {}

for i, ticker in enumerate(tickers, 1):
    try:
        df = yf.download(
            ticker,
            start="2021-01-01",
            end="2025-01-01",
            auto_adjust=False,
            actions=True,
            progress=False,
            threads=False
        )

        if df.empty:
            download_failed[ticker] = "No data returned"
            continue

        # Flatten single-ticker MultiIndex columns
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df.index = pd.to_datetime(df.index).tz_localize(None)
        df.index.name = "Date"

        stock_data[ticker] = df

    except Exception as e:
        download_failed[ticker] = str(e)

    if i % 50 == 0:
        print(f"Processed {i}/{len(tickers)} tickers")

print("Successfully downloaded:", len(stock_data))
print("Failed downloads:", len(download_failed))

Processed 50/503 tickers
Processed 100/503 tickers
Processed 150/503 tickers


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['FDXF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Processed 200/503 tickers


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['HONA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Processed 250/503 tickers
Processed 300/503 tickers
Processed 350/503 tickers


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['Q']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Processed 400/503 tickers


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['SNDK']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Processed 450/503 tickers


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['VYLR']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Processed 500/503 tickers
Successfully downloaded: 498
Failed downloads: 5


In [9]:
stock_data.get("AAPL").head()

Price,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume
Date,,,,,,,,
2021-01-04,125.537125,129.410004,0.0,133.610001,126.760002,133.520004,0.0,143301900
2021-01-05,127.089226,131.009995,0.0,131.740005,128.429993,128.889999,0.0,97664900
2021-01-06,122.811218,126.599998,0.0,131.050003,126.379997,127.720001,0.0,155088000
2021-01-07,127.001938,130.919998,0.0,131.630005,127.860001,128.360001,0.0,109578200
2021-01-08,128.098099,132.050003,0.0,132.630005,130.229996,132.429993,0.0,105158200


In [10]:
benchmark = yf.download(
    "^GSPC",
    start="2021-01-01",
    end="2025-01-01",
    auto_adjust=False,
    actions=True,
    progress=False
)

In [11]:
if isinstance(benchmark.columns, pd.MultiIndex):
    benchmark.columns = benchmark.columns.get_level_values(0)


benchmark.index = pd.to_datetime(benchmark.index).tz_localize(None)
benchmark.index.name = "Date"

benchmark_df = benchmark.to_csv("sp500_data/sp500_benchmark.csv")

display(benchmark.head())

Price,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume
Date,,,,,,,,
2021-01-04,3700.649902,3700.649902,0.0,3769.989990,3662.709961,3764.610107,0.0,5015000000
2021-01-05,3726.860107,3726.860107,0.0,3737.830078,3695.070068,3698.020020,0.0,4591020000
2021-01-06,3748.139893,3748.139893,0.0,3783.040039,3705.340088,3712.199951,0.0,6064110000
2021-01-07,3803.790039,3803.790039,0.0,3811.550049,3764.709961,3764.709961,0.0,5099160000
2021-01-08,3824.679932,3824.679932,0.0,3826.689941,3783.600098,3815.050049,0.0,4773040000


In [12]:
all_stocks = []

for ticker, df in stock_data.items():
    temp = df.reset_index().copy()
    temp["Ticker"] = ticker
    all_stocks.append(temp)

if not all_stocks:
    raise RuntimeError("No stock data was downloaded.")

daily_data = pd.concat(all_stocks, ignore_index=True)

daily_data.to_csv(
    "sp500_data/sp500_daily_prices.csv",
    index=False
)

print("Total daily records:", len(daily_data))
print("Total downloaded tickers:", daily_data["Ticker"].nunique())

display(daily_data.head())

Total daily records: 496304
Total downloaded tickers: 498


Price,Date,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume,Ticker
0,2021-01-04,117.366699,143.704010,0.0,147.324417,142.600327,146.321075,0.0,3583455,MMM
1,2021-01-05,117.168648,143.461533,0.0,144.857864,142.683945,143.821075,0.0,2745179,MMM
2,2021-01-06,118.950943,145.643814,0.0,146.797653,143.846161,144.414719,0.0,4002294,MMM
3,2021-01-07,115.898506,141.906357,0.0,145.033447,138.929764,143.444809,0.0,7012626,MMM
4,2021-01-08,113.781563,139.314377,0.0,141.755859,137.633774,141.446487,0.0,5750488,MMM


In [13]:
download_audit = pd.DataFrame([
    {
        "Ticker": ticker,
        "Download_Status": (
            "Downloaded" if ticker in stock_data else "Failed"
        ),
        "Reason": download_failed.get(ticker, "")
    }
    for ticker in tickers
])

download_audit.to_csv(
    "sp500_data/download_audit.csv",
    index=False
)

display(download_audit["Download_Status"].value_counts())

,count
Download_Status,
Downloaded,498
Failed,5


In [14]:
display(download_audit[download_audit["Download_Status"] == "Failed"])

,Ticker,Download_Status,Reason
192,FDXF,Failed,No data returned
233,HONA,Failed,No data returned
379,Q,Failed,No data returned
402,SNDK,Failed,No data returned
478,VYLR,Failed,No data returned


In [15]:
START_DATE = "2021-01-01"
END_DATE = "2025-01-01"

RETRIEVAL_DATE = datetime.now(timezone.utc).strftime(
    "%Y-%m-%d %H:%M:%S UTC"
)

metadata = {
    "ticker_source": url,
    "retrieval_date": RETRIEVAL_DATE,
    "data_provider": "Yahoo Finance",
    "python_library": "yfinance",
    "start_date": START_DATE,
    "end_date_exclusive": END_DATE,
    "benchmark": "^GSPC",
    "requested_fields": [
        "Open", "High", "Low", "Close", "Adj Close", "Volume"
    ],
    "total_constituents": len(tickers),
    "successful_downloads": len(stock_data),
    "failed_downloads": len(download_failed)
}

with open("sp500_data/acquisition_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Acquisition metadata saved.")

Acquisition metadata saved.


## Task 1.2

In [16]:
# Reload saved data so the rest of the notebook works after a Colab restart
import pandas as pd
import numpy as np
import os

daily_data = pd.read_csv('sp500_data/sp500_daily_prices.csv', parse_dates=['Date'])
benchmark = pd.read_csv('sp500_data/sp500_benchmark.csv', parse_dates=['Date'])
daily_data['Date'] = pd.to_datetime(daily_data['Date']).dt.tz_localize(None).dt.normalize()
benchmark['Date'] = pd.to_datetime(benchmark['Date']).dt.tz_localize(None).dt.normalize()

print('Stock dates:', daily_data['Date'].min(), 'to', daily_data['Date'].max())
print('Benchmark dates:', benchmark['Date'].min(), 'to', benchmark['Date'].max())
print('Downloaded stocks:', daily_data['Ticker'].nunique())

Stock dates: 2021-01-04 00:00:00 to 2024-12-31 00:00:00
Benchmark dates: 2021-01-04 00:00:00 to 2024-12-31 00:00:00
Downloaded stocks: 498


In [17]:
# Check duplicate trading dates
stock_duplicates = daily_data.duplicated(['Ticker', 'Date'], keep=False)
benchmark_duplicates = benchmark.duplicated('Date', keep=False)
print('Stock duplicate rows:', stock_duplicates.sum())
print('Benchmark duplicate rows:', benchmark_duplicates.sum())
display(daily_data.loc[stock_duplicates].head())

Stock duplicate rows: 0
Benchmark duplicate rows: 0


,Date,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume,Ticker


In [18]:
# Check missing values and invalid prices / volumes
price_columns = ['Open', 'High', 'Low', 'Close', 'Adj Close']
required_columns = price_columns + ['Volume']
for col in required_columns:
    daily_data[col] = pd.to_numeric(daily_data[col], errors='coerce')
    benchmark[col] = pd.to_numeric(benchmark[col], errors='coerce')

missing_summary = daily_data.groupby('Ticker')[required_columns].apply(lambda x: x.isna().sum())
missing_summary['Total_Missing'] = missing_summary.sum(axis=1)
daily_data['Invalid_Price'] = daily_data[price_columns].le(0).any(axis=1)
daily_data['Invalid_Volume'] = (~np.isfinite(daily_data['Volume'])) | (daily_data['Volume'] < 0)
daily_data['Zero_Volume'] = daily_data['Volume'].eq(0)
print('Stocks with missing fields:', (missing_summary['Total_Missing'] > 0).sum())
print('Nonpositive price rows:', daily_data['Invalid_Price'].sum())
print('Invalid volume rows:', daily_data['Invalid_Volume'].sum())
print('Zero-volume rows to investigate:', daily_data['Zero_Volume'].sum())
display(missing_summary[missing_summary['Total_Missing'] > 0].head())

Stocks with missing fields: 0
Nonpositive price rows: 0
Invalid volume rows: 0
Zero-volume rows to investigate: 451


,Open,High,Low,Close,Adj Close,Volume,Total_Missing
Ticker,,,,,,,


In [19]:
# Compare stock histories
history_summary = daily_data.groupby('Ticker').agg(
    First_Date=('Date','min'), Last_Date=('Date','max'),
    Trading_Days=('Date','nunique')
).reset_index()
display(history_summary.sort_values('Trading_Days').head(15))

,Ticker,First_Date,Last_Date,Trading_Days
201,GEV,2024-03-27,2024-12-31,193
407,SOLV,2024-03-26,2024-12-31,194
382,RDDT,2024-03-21,2024-12-31,197
463,VLTO,2023-10-04,2024-12-31,313
199,GEHC,2022-12-15,2024-12-31,513
84,CEG,2022-01-19,2024-12-31,742
224,HOOD,2021-07-29,2024-12-31,862
39,APP,2021-04-15,2024-12-31,935
104,COIN,2021-04-14,2024-12-31,936
171,EXE,2021-02-10,2024-12-31,979


In [20]:
# Check recorded splits and dividends; missing action columns mean unknown, not none
for col in ['Dividends', 'Stock Splits']:
    if col not in daily_data.columns:
        daily_data[col] = np.nan
corporate_actions = daily_data.loc[
    daily_data['Dividends'].fillna(0).ne(0) |
    daily_data['Stock Splits'].fillna(0).ne(0),
    ['Ticker','Date','Dividends','Stock Splits']
]
print('Recorded corporate-action rows:', len(corporate_actions))
display(corporate_actions.head(10))

Recorded corporate-action rows: 6147


,Ticker,Date,Dividends,Stock Splits
27,MMM,2021-02-11,1.237458,0.0
95,MMM,2021-05-20,1.237458,0.0
159,MMM,2021-08-20,1.237458,0.0
222,MMM,2021-11-18,1.237458,0.0
284,MMM,2022-02-17,1.245819,0.0
347,MMM,2022-05-19,1.245819,0.0
410,MMM,2022-08-19,1.245819,0.0
473,MMM,2022-11-17,1.245819,0.0
534,MMM,2023-02-16,1.254181,0.0
597,MMM,2023-05-18,1.254181,0.0


In [21]:
# Record exclusions. Shorter histories alone are not grounds for exclusion.
audit = daily_data.groupby('Ticker').agg(
    Duplicate_Dates=('Date', lambda s: s.duplicated(keep=False).sum()),
    Invalid_Prices=('Invalid_Price','sum'),
    Invalid_Volume=('Invalid_Volume','sum'),
    Zero_Volume_Days=('Zero_Volume','sum'),
    Trading_Days=('Date','nunique')
).join(missing_summary['Total_Missing'].rename('Missing_Values')).reset_index()

def reason(row):
    issues = []
    if row['Duplicate_Dates']: issues.append('Duplicate dates')
    if row['Invalid_Prices']: issues.append('Nonpositive prices')
    if row['Invalid_Volume']: issues.append('Invalid volume')
    if row['Missing_Values']: issues.append('Missing values')
    return '; '.join(issues)

audit['Exclusion_Reason'] = audit.apply(reason, axis=1)
audit['Status'] = np.where(audit['Exclusion_Reason'].eq(''), 'Valid', 'Excluded')
valid_tickers = audit.loc[audit['Status'] == 'Valid', 'Ticker'].tolist()
excluded_stocks = audit.loc[audit['Status'] == 'Excluded'].copy()
valid_data = daily_data[daily_data['Ticker'].isin(valid_tickers)].copy()

# Failed downloads from Task 1.1 are also recorded separately.
if os.path.exists('sp500_data/download_audit.csv'):
    download_audit = pd.read_csv('sp500_data/download_audit.csv')
    failed_downloads = download_audit[download_audit['Download_Status'] == 'Failed'].copy()
else:
    failed_downloads = pd.DataFrame()

print('Valid shares:', len(valid_tickers), 'Excluded for quality:', len(excluded_stocks))
print('Failed downloads:', len(failed_downloads))
display(excluded_stocks.head())

Valid shares: 498 Excluded for quality: 0
Failed downloads: 5


,Ticker,Duplicate_Dates,Invalid_Prices,Invalid_Volume,Zero_Volume_Days,Trading_Days,Missing_Values,Exclusion_Reason,Status


In [22]:
# Verify benchmark data before resampling
if benchmark['Date'].duplicated().any():
    raise ValueError('Duplicate benchmark dates: investigate before continuing')
if (benchmark[required_columns].isna().any(axis=1) |
    benchmark[price_columns].le(0).any(axis=1) |
    (benchmark['Volume'] < 0)).any():
    raise ValueError('Invalid benchmark data: investigate before continuing')
if valid_data.empty:
    raise ValueError('No validated stock data remains')

In [23]:
# Friday-ending adjusted closing prices
# Use date as index, not ticker, before resampling
weekly_stocks = (
    valid_data.sort_values(['Ticker','Date'])
    .set_index('Date')
    .groupby('Ticker')['Adj Close']
    .resample('W-FRI').last()
    .unstack('Ticker')
)
weekly_benchmark = (
    benchmark.sort_values('Date')
    .set_index('Date')['Adj Close']
    .resample('W-FRI').last()
)

# Exclude incomplete first and final calendar weeks
weekly_stocks = weekly_stocks.loc['2021-01-08':'2024-12-27']
weekly_benchmark = weekly_benchmark.loc['2021-01-08':'2024-12-27']

print('Stock weekly shape:', weekly_stocks.shape)
print('Benchmark weekly rows:', len(weekly_benchmark))
if weekly_stocks.empty or weekly_benchmark.empty:
    raise ValueError('Empty weekly prices. Inspect the daily date ranges above.')

Stock weekly shape: (208, 498)
Benchmark weekly rows: 208


In [24]:
# Align both price series on the same Friday labels
common_weeks = weekly_stocks.index.intersection(weekly_benchmark.index)
weekly_stocks = weekly_stocks.reindex(common_weeks)
weekly_benchmark = weekly_benchmark.reindex(common_weeks)

# Keep missing stock prices as NaN. Do not fill them.
print('Common Friday labels:', len(common_weeks))
print('Indices match:', weekly_stocks.index.equals(weekly_benchmark.index))
display(weekly_stocks.head())
display(weekly_benchmark.head())

Common Friday labels: 208
Indices match: True


Ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2021-01-08,122.100449,128.098099,85.705215,149.770004,99.498413,34.650742,238.995819,485.100006,142.359100,44.473660,...,26.578350,104.901443,54.840778,36.195755,100.191071,241.449997,96.210403,146.956482,405.470001,157.178085
2021-01-15,121.283630,123.335030,89.334625,169.270004,99.621773,32.986668,230.271225,458.079987,143.276398,44.160637,...,25.893581,102.207474,54.173611,38.130554,96.570808,227.750000,95.734520,148.038193,409.220001,151.054001
2021-01-22,121.398949,134.908005,89.609444,181.869995,101.000198,31.674431,231.288040,472.440002,141.305481,44.685154,...,26.442989,103.381271,54.006824,37.764305,95.731064,222.880005,96.156525,148.861038,408.839996,152.945602
2021-01-29,115.479385,128.010834,82.835800,183.630005,110.622223,29.867723,219.622391,458.769989,133.812439,42.307926,...,24.834591,95.761162,53.364689,35.702114,90.123436,215.960007,91.128258,142.074844,387.829987,144.442917
2021-02-05,118.371933,132.865433,87.887756,195.309998,111.016090,32.140369,230.171387,492.119995,135.111221,45.091225,...,27.390512,112.560028,52.906010,39.770756,91.616318,240.380005,94.585167,147.825531,407.109985,149.153061


,Adj Close
Date,
2021-01-08,3824.679932
2021-01-15,3768.250000
2021-01-22,3841.469971
2021-01-29,3714.239990
2021-02-05,3886.830078


In [25]:
audit.to_csv('sp500_data/validation_audit.csv', index=False)
excluded_stocks.to_csv('sp500_data/excluded_stocks.csv', index=False)
failed_downloads.to_csv('sp500_data/failed_downloads.csv', index=False)
valid_data.to_csv('sp500_data/validated_daily_data.csv', index=False)
weekly_stocks.to_csv('sp500_data/weekly_adjusted_prices.csv')
weekly_benchmark.to_csv('sp500_data/weekly_benchmark_prices.csv')

## Task 1.3
### 1.3.1 Mean weekly return

In [26]:
# Adjacent weekly prices only: missing weeks are not filled
stock_weekly_returns = weekly_stocks / weekly_stocks.shift(1) - 1
benchmark_weekly_returns = weekly_benchmark / weekly_benchmark.shift(1) - 1
print('Weekly stock returns:')
display(stock_weekly_returns.head(10))
print('Benchmark weekly returns:')
display(benchmark_weekly_returns.head(10))

Weekly stock returns:


Ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2021-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-01-15,-0.006690,-0.037183,0.042348,0.130200,0.001240,-0.048024,-0.036505,-0.055700,0.006444,-0.007038,...,-0.025764,-0.025681,-0.012166,0.053454,-0.036134,-0.056741,-0.004946,0.007361,0.009249,-0.038963
2021-01-22,0.000951,0.093834,0.003076,0.074437,0.013837,-0.039781,0.004416,0.031348,-0.013756,0.011877,...,0.021218,0.011484,-0.003079,-0.009605,-0.008696,-0.021383,0.004408,0.005558,-0.000929,0.012523
2021-01-29,-0.048761,-0.051125,-0.075591,0.009677,0.095267,-0.057040,-0.050438,-0.028935,-0.053027,-0.053199,...,-0.060825,-0.073709,-0.011890,-0.054607,-0.058577,-0.031048,-0.052293,-0.045587,-0.051389,-0.055593
2021-02-05,0.025048,0.037923,0.060988,0.063606,0.003560,0.076090,0.048032,0.072694,0.009706,0.065787,...,0.102918,0.175425,-0.008595,0.113961,0.016565,0.113076,0.037935,0.040476,0.049713,0.032609
2021-02-12,0.038967,-0.010164,-0.039455,0.088936,0.033863,0.039941,0.015500,0.013655,0.080801,0.052711,...,0.018604,0.007351,-0.029950,0.028587,0.009472,0.134662,0.006033,0.009319,0.173491,0.046647
2021-02-19,-0.008673,-0.040629,0.005457,-0.054589,-0.040474,-0.001707,-0.012351,-0.039532,0.018783,0.007179,...,-0.001427,0.049300,0.008612,0.036619,0.009109,0.014006,-0.003224,-0.029186,0.027630,-0.040249
2021-02-26,-0.037834,-0.066297,0.025998,0.026260,-0.026495,0.020804,-0.013331,-0.040595,-0.044642,0.008197,...,-0.032009,0.065260,-0.056066,0.038190,-0.001804,-0.168276,-0.015219,0.040787,0.017293,-0.029750
2021-03-05,-0.048579,0.001319,-0.009653,-0.128616,-0.021122,0.015355,-0.005540,-0.040986,-0.044539,0.017850,...,0.001832,0.012070,0.047107,0.120655,0.006830,-0.059079,0.007727,-0.027966,-0.042609,-0.061582


Benchmark weekly returns:


,Adj Close
Date,
2021-01-08,NaN
2021-01-15,-0.014754
2021-01-22,0.019431
2021-01-29,-0.033120
2021-02-05,0.046467
2021-02-12,0.012349
2021-02-19,-0.007146
2021-02-26,-0.024460
2021-03-05,0.008079


In [27]:
# Mean weekly returns from observations aligned with the benchmark
aligned_returns = stock_weekly_returns.loc[benchmark_weekly_returns.notna()]
mean_weekly_returns = aligned_returns.mean(axis=0, skipna=True)
print('Mean weekly returns:')
display(mean_weekly_returns.rename('Mean_Weekly_Return').head(10))

Mean weekly returns:


,Mean_Weekly_Return
Ticker,
A,0.001121
AAPL,0.003930
ABBV,0.003749
ABNB,0.001475
ABT,0.000971
ACGL,0.005289
ACN,0.002376
ADBE,0.000990
ADI,0.002610


In [37]:
print('Mean of Mean Weekly Returns:', mean_weekly_returns.mean())
print('Median of Mean Weekly Returns:', mean_weekly_returns.median())
print('Max of Mean Weekly Returns:', mean_weekly_returns.max())
print('Min of Mean Weekly Returns:', mean_weekly_returns.min())

Mean of Mean Weekly Returns: 0.003034842063075907
Median of Mean Weekly Returns: 0.002688994257725265
Max of Mean Weekly Returns: 0.038490943011959794
Min of Mean Weekly Returns: -0.004468092375247928


### 1.3.2 Weekly beta

In [28]:
# Covariance(stock, market) / variance(market), using matching week labels
beta_results = []
for ticker in stock_weekly_returns.columns:
    paired = pd.concat([
        stock_weekly_returns[ticker].rename('Stock'),
        benchmark_weekly_returns.rename('Market')
    ], axis=1).dropna()
    n = len(paired)
    market_variance = paired['Market'].var() if n >= 2 else np.nan
    beta = (paired['Stock'].cov(paired['Market']) / market_variance
            if pd.notna(market_variance) and market_variance > 0 else np.nan)
    beta_results.append({
        'Ticker': ticker,
        'Mean_Weekly_Return': paired['Stock'].mean() if n else np.nan,
        'Weekly_Beta': beta,
        'Aligned_Weekly_Observations': n
    })
weekly_features = pd.DataFrame(beta_results)
weekly_features.to_csv('sp500_data/weekly_features.csv', index=False)
display(weekly_features.head(10))
print('Valid beta values:', weekly_features['Weekly_Beta'].notna().sum())

,Ticker,Mean_Weekly_Return,Weekly_Beta,Aligned_Weekly_Observations
0,A,0.001121,1.074724,207
1,AAPL,0.003930,1.174835,207
2,ABBV,0.003749,0.425366,207
3,ABNB,0.001475,1.472709,207
4,ABT,0.000971,0.583543,207
5,ACGL,0.005289,0.586638,207
6,ACN,0.002376,1.143300,207
7,ADBE,0.000990,1.553525,207
8,ADI,0.002610,1.131967,207
9,ADM,0.001048,0.548876,207


Valid beta values: 498


In [38]:
print('Mean of Weekly Beta:', weekly_features['Weekly_Beta'].mean())
print('Median of Weekly Beta:', weekly_features['Weekly_Beta'].median())
print('Max of Weekly Beta:', weekly_features['Weekly_Beta'].max())
print('Min of Weekly Beta:', weekly_features['Weekly_Beta'].min())

Mean of Weekly Beta: 0.9841620032331683
Median of Weekly Beta: 0.9547121937039504
Max of Weekly Beta: 3.778559682265264
Min of Weekly Beta: 0.05545295020774641


## Task 1.4
### Adapted RFM calculations

In [29]:
# Use the final completed benchmark trading date also present for each RFM share.
# Shares must have 80 consecutive benchmark sessions: 20 for thresholds + final 60.
benchmark_sessions = pd.DatetimeIndex(
    benchmark.loc[(benchmark['Date'] >= '2021-01-01') &
                  (benchmark['Date'] < '2025-01-01'), 'Date'].sort_values().unique()
)

rfm_source = valid_data.copy()
rfm_source = rfm_source[rfm_source['Date'].isin(benchmark_sessions)]

# Find the latest date shared by all currently validated shares and the benchmark
stock_date_sets = [set(g['Date']) for _,g in rfm_source.groupby('Ticker')]
common_dates = set(benchmark_sessions)
for dates in stock_date_sets:
    common_dates.intersection_update(dates)
if not common_dates:
    raise ValueError('No common observation date across validated shares')
observation_date = max(common_dates)
print('Observation date:', observation_date.date())

window_dates = benchmark_sessions[benchmark_sessions <= observation_date][-80:]
if len(window_dates) < 80:
    raise ValueError('Fewer than 80 benchmark sessions before observation date')

# Each retained share must have valid observations on every one of the 80 sessions
window_counts = (rfm_source[rfm_source['Date'].isin(window_dates)]
                 .groupby('Ticker')['Date'].nunique())
rfm_tickers = window_counts[window_counts == 80].index.tolist()
rfm_window_exclusions = pd.DataFrame({
    'Ticker': sorted(set(valid_tickers) - set(rfm_tickers)),
    'Reason': 'Missing benchmark-aligned session(s) in final 80-session window'
})
print('RFM eligible shares:', len(rfm_tickers))
print('RFM window exclusions:', len(rfm_window_exclusions))

Observation date: 2024-12-31
RFM eligible shares: 498
RFM window exclusions: 0


In [30]:
# Rolling mean uses the previous 20 trading sessions, not today's volume
rfm_daily = rfm_source[rfm_source['Ticker'].isin(rfm_tickers)].copy()
rfm_daily = rfm_daily.sort_values(['Ticker','Date'])
rfm_daily['Previous_20_Avg_Volume'] = (
    rfm_daily.groupby('Ticker')['Volume']
    .transform(lambda x: x.shift(1).rolling(20, min_periods=20).mean())
)
rfm_daily['High_Volume_Event'] = (
    (rfm_daily['Volume'] > 2 * rfm_daily['Previous_20_Avg_Volume']) &
    rfm_daily['Previous_20_Avg_Volume'].notna()
)
latest_60_dates = window_dates[-60:]
latest_60 = rfm_daily[rfm_daily['Date'].isin(latest_60_dates)].copy()
print('Sessions per share:', latest_60.groupby('Ticker').size().unique())

Sessions per share: [60]


### 1.4.1 Recency

In [31]:
last_event = (rfm_daily[rfm_daily['High_Volume_Event']]
              .groupby('Ticker')['Date'].max())
rfm_features = pd.DataFrame({'Ticker': rfm_tickers})
rfm_features['Last_High_Volume_Date'] = rfm_features['Ticker'].map(last_event)
rfm_features['Recency'] = (
    observation_date - rfm_features['Last_High_Volume_Date']
).dt.days
print('No qualifying event:', rfm_features['Recency'].isna().sum())
display(rfm_features.head())

No qualifying event: 0


,Ticker,Last_High_Volume_Date,Recency
0,A,2024-12-20,11
1,AAPL,2024-12-20,11
2,ABBV,2024-12-20,11
3,ABNB,2024-12-20,11
4,ABT,2024-12-20,11


### 1.4.2 Frequency

In [32]:
frequency = latest_60.groupby('Ticker')['High_Volume_Event'].sum().astype(int)
rfm_features['Frequency'] = rfm_features['Ticker'].map(frequency)
display(rfm_features[['Ticker','Recency','Frequency']].head())

,Ticker,Recency,Frequency
0,A,11,2
1,AAPL,11,2
2,ABBV,11,3
3,ABNB,11,3
4,ABT,11,1


### 1.4.3 Monetary

In [33]:
latest_60['Daily_Traded_Value'] = latest_60['Close'] * latest_60['Volume']
monetary = latest_60.groupby('Ticker')['Daily_Traded_Value'].sum(min_count=60)
rfm_features['Monetary'] = rfm_features['Ticker'].map(monetary)
display(rfm_features.head(10))

,Ticker,Last_High_Volume_Date,Recency,Frequency,Monetary
0,A,2024-12-20,11,2,1.400180e+10
1,AAPL,2024-12-20,11,2,6.233034e+11
2,ABBV,2024-12-20,11,3,6.837054e+10
3,ABNB,2024-12-20,11,3,3.652850e+10
4,ABT,2024-12-20,11,1,3.523027e+10
5,ACGL,2024-12-20,11,4,1.196213e+10
6,ACN,2024-12-20,11,2,5.199430e+10
7,ADBE,2024-12-13,18,3,1.030403e+11
8,ADI,2024-11-25,36,1,3.849918e+10
9,ADM,2024-12-20,11,3,9.529780e+09


In [34]:
# Save RFM and final clustering features
rfm_clustering_ready = rfm_features.dropna(subset=['Recency','Frequency','Monetary']).copy()
no_event_exclusions = rfm_features[rfm_features['Recency'].isna()].copy()
no_event_exclusions['Reason'] = 'No qualifying high-volume event'
rfm_exclusions = pd.concat([
    rfm_window_exclusions,
    no_event_exclusions[['Ticker','Reason']]
], ignore_index=True)
final_features = weekly_features.merge(
    rfm_clustering_ready[['Ticker','Recency','Frequency','Monetary']],
    on='Ticker', how='inner'
).dropna(subset=['Mean_Weekly_Return','Weekly_Beta'])

rfm_features.to_csv('sp500_data/rfm_features.csv', index=False)
rfm_clustering_ready.to_csv('sp500_data/rfm_clustering_ready.csv', index=False)
rfm_exclusions.to_csv('sp500_data/rfm_exclusions.csv', index=False)
latest_60.to_csv('sp500_data/latest_60_sessions.csv', index=False)
final_features.to_csv('sp500_data/task1_final_features.csv', index=False)

print('Final observation date:', observation_date.date())
print('Weekly feature stocks:', len(weekly_features))
print('RFM eligible stocks:', len(rfm_features))
print('Final clustering-ready stocks:', len(final_features))
display(final_features.head(10))

Final observation date: 2024-12-31
Weekly feature stocks: 498
RFM eligible stocks: 498
Final clustering-ready stocks: 498


,Ticker,Mean_Weekly_Return,Weekly_Beta,Aligned_Weekly_Observations,Recency,Frequency,Monetary
0,A,0.001121,1.074724,207,11,2,1.400180e+10
1,AAPL,0.003930,1.174835,207,11,2,6.233034e+11
2,ABBV,0.003749,0.425366,207,11,3,6.837054e+10
3,ABNB,0.001475,1.472709,207,11,3,3.652850e+10
4,ABT,0.000971,0.583543,207,11,1,3.523027e+10
5,ACGL,0.005289,0.586638,207,11,4,1.196213e+10
6,ACN,0.002376,1.143300,207,11,2,5.199430e+10
7,ADBE,0.000990,1.553525,207,18,3,1.030403e+11
8,ADI,0.002610,1.131967,207,36,1,3.849918e+10
9,ADM,0.001048,0.548876,207,11,3,9.529780e+09
